In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import time
import torch
import pandas as pd

import pygeohash as pgh

from collections import defaultdict
from torch_geometric.utils.convert import from_networkx
from torch_geometric.utils import to_undirected


%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import torch
import random

def set_seed(seed=42):
    """Set the same seed for reproducibility across NumPy, PyTorch, and Python."""
    np.random.seed(seed)             # NumPy
    torch.manual_seed(seed)          # PyTorch CPU
    torch.cuda.manual_seed(seed)     # PyTorch GPU (if applicable)
    torch.cuda.manual_seed_all(seed) # If using multiple GPUs
    random.seed(seed)                # Python's built-in random module

    # Ensure deterministic behavior in PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
set_seed(seed=42)

In [ ]:
base_path = '../data'

In [ ]:
df_checkin = pd.read_csv(f"{base_path}/raw/loc-gowalla_totalCheckins.txt", delimiter="\t", names=["user", "check-in-time", "latitude", "longitude", "location_id"])
df_friend = pd.read_csv(f"{base_path}/raw/loc-gowalla_edges.txt", delimiter="\t", names=["src", "dst"])

In [ ]:
df_checkin["geohash"] = df_checkin.apply(lambda row: pgh.encode(row["latitude"], row["longitude"], precision=8), axis=1)


In [ ]:
df_checkin["datetime"] = pd.to_datetime(df_checkin["check-in-time"])
df_checkin["timestamp"] = df_checkin["datetime"].astype(int) // 10**9  # Convert nanoseconds to seconds


In [ ]:
df_checkin["event_set"] = 'personal'
df_checkin["event"] = df_checkin['geohash']
df_checkin["uid"] = df_checkin['user']
df_checkin["other_uid"] = None

In [ ]:
dfp = df_checkin[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].sort_values(by=["uid", "timestamp"]).reset_index(drop=True)

In [ ]:
df_checkin

In [ ]:
dfp

In [ ]:
edge_index = torch.tensor(df_friend[['src', 'dst']].values.T, dtype=torch.long)  # Shape: (2, num_edges)
edge_index = to_undirected(edge_index)
df_undirected = pd.DataFrame(edge_index.T.numpy(), columns=["src", "dst"])

In [ ]:
df_undirected["timestamp"] = None
df_undirected["event_set"] = 'relational'
df_undirected["event"] = 'friendship'
df_undirected["uid"] = df_undirected['src']
df_undirected["other_uid"] = df_undirected['dst']

In [ ]:
dfr = df_undirected[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].sort_values(by=["uid", "timestamp"]).reset_index(drop=True)

In [ ]:
df_friend

In [ ]:
dfr

In [ ]:
df_all = pd.concat([dfp, dfr]).drop_duplicates().sort_values(by=["uid", "timestamp", 'other_uid']).reset_index(drop=True)
df_all

In [ ]:
df_stats = df_all.groupby(['uid', 'event_set'])['event'].count().reset_index().sort_values('event')
df_stats

In [ ]:
dfg = df_stats.groupby(['uid'])['event_set'].count().reset_index()
id_have_2set = dfg[dfg['event_set'] == 2]['uid'].values
id_have_2set

In [ ]:
df_all

In [ ]:
df_final = df_all[df_all['uid'].isin(id_have_2set)].reset_index(drop=True)
df_final

In [ ]:
df_stats

In [ ]:
df_stats.groupby(['event_set'])['event'].sum().reset_index()

In [ ]:
df_final.groupby(['uid', 'event_set'])['event'].count().reset_index().sort_values('event').groupby(['event_set'])['event'].sum().reset_index()

In [ ]:
df_all.to_csv(f'{base_path}/processed/gowalla_all_events.csv', index=False)


In [ ]:
df_final.to_csv(f'{base_path}/processed/gowalla_accounts_with_pr.csv', index=False)
